In [141]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

In [142]:
df = pd.read_csv("HousePrice.csv")

In [143]:
print("\n--- Current Missing Data Proportions (%) ---")
print(df.isnull().mean() * 100)


--- Current Missing Data Proportions (%) ---
id                       0.0
area                     0.0
area_sqft                0.0
lot_size_sqft            0.0
bedrooms                 0.0
bathrooms                0.0
stories                  0.0
parking_spaces           0.0
house_age_years          0.0
distance_to_city_km      3.0
school_quality_index    22.0
crime_index             12.0
price                    0.0
dtype: float64


In [144]:
print("--- Total Missing Rows Before Cleaning ---")
print(df.isnull().sum())

--- Total Missing Rows Before Cleaning ---
id                          0
area                        0
area_sqft                   0
lot_size_sqft               0
bedrooms                    0
bathrooms                   0
stories                     0
parking_spaces              0
house_age_years             0
distance_to_city_km      1500
school_quality_index    11000
crime_index              6000
price                       0
dtype: int64


In [145]:
# PHASE 1: SIMPLE & DIRECT INPUT FIDELITY

In [146]:
# TASK 1: MISSING VALUE BRACKETS (DECISION MATRIX)

In [147]:
df = df.dropna(subset=["distance_to_city_km"])

In [148]:
crime_median = df["crime_index"].median()
df["crime_index"] = df["crime_index"].fillna(crime_median)

In [149]:
knn = KNNImputer(n_neighbors=5)
df[["school_quality_index"]] = knn.fit_transform(df[["school_quality_index"]])

In [150]:
# TASK 2: PURE VECTORIZED OUTLIER HANDLING

In [151]:
df_numeric = df[[
    "area_sqft", 
    "lot_size_sqft", 
    "distance_to_city_km", 
    "school_quality_index", 
    "crime_index", 
    "price"
]]

In [152]:
# Vectorized mathematical operations

Q1 = df_numeric.quantile(0.25)
Q3 = df_numeric.quantile(0.75)
IQR = Q3 - Q1

In [153]:
#Non-parametric statistical boundaries calculate

lower_bounds = Q1 - 1.5 * IQR
upper_bounds = Q3 + 1.5 * IQR

In [154]:
# Pure Vectorized Winsorization via numpy.clip()

df[[
    "area_sqft", 
    "lot_size_sqft", 
    "distance_to_city_km", 
    "school_quality_index", 
    "crime_index", 
    "price"
]] = df_numeric.clip(lower=lower_bounds, upper=upper_bounds, axis=1)

print("Success! Vectorized outlier capping completed")

Success! Vectorized outlier capping completed


In [155]:
# PHASE 2: THE VECTORIZED COMPUTATION ENGINE

In [156]:
# TASK 1: FEATURE ENGINEERING   

In [157]:
df["total_rooms"] = df["bedrooms"] + df["bathrooms"]
df["yard_size_sqft"] = df["lot_size_sqft"] - df["area_sqft"]
df["space_per_room"] = df["area_sqft"] / (df["total_rooms"] + 1)

print("Task 1 Complete: 3 New predictive features engineered successfully!")

Task 1 Complete: 3 New predictive features engineered successfully!


In [158]:
df

,id,area,area_sqft,lot_size_sqft,bedrooms,bathrooms,stories,parking_spaces,house_age_years,distance_to_city_km,school_quality_index,crime_index,price,total_rooms,yard_size_sqft,space_per_room
0,1,PECHS,2098,3246,4,4,3,1,19,3.73,3.630000,2.21,468238.0,8,1148,233.111111
1,2,NorthNazimabad,1717,3041,3,2,2,1,25,6.48,7.400000,3.88,391372.0,5,1324,286.166667
2,3,PECHS,2189,6396,4,3,2,1,10,2.37,6.952174,7.71,582627.0,7,4207,273.625000
3,4,PECHS,2714,7164,6,5,2,2,10,18.56,8.770000,3.55,695062.0,11,4450,226.166667
5,6,PECHS,1660,4865,3,2,1,0,8,1.63,6.800000,5.62,416013.0,5,3205,276.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,49996,FB Area,1834,2519,3,3,2,1,46,24.36,8.800000,5.63,281660.0,6,685,262.000000
49996,49997,NayaNazimabad,1785,6001,3,2,1,4,3,2.56,8.090000,4.90,442028.0,5,4216,297.500000
49997,49998,FB Area,2100,6910,4,3,2,1,19,2.50,6.952174,2.51,408194.0,7,4810,262.500000
49998,49999,NorthNazimabad,1959,3164,3,2,1,0,40,18.06,6.030000,4.98,529438.0,5,1205,326.500000


In [159]:
# TASK 2: One-Hot Encoding

df = pd.get_dummies(df, columns=["area"], drop_first=True, dtype=int)

print("Task 2 Complete: 'area' encoded into equidistant coordinate axes!")

Task 2 Complete: 'area' encoded into equidistant coordinate axes!


In [160]:
# TASK 3: COLLINEARITY ERADICATION ALGORITHM

numeric_features = df.select_dtypes(include=[np.number]).drop(columns=["id"])

# Step 1: Build Absolute Matrix
correlation_matrix = numeric_features.corr().abs()


In [161]:
# Step 2: Isolate Upper Triangle
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

In [162]:
# Step 3: Identify Pairs > 0.80
high_corr_pairs = [
    (column, upper_triangle[column].idxmax())
    for column in upper_triangle.columns
    if upper_triangle[column].max() > 0.80
]
print(f"\nHighly Collinear Pairs Identified (>0.80): {high_corr_pairs}")


Highly Collinear Pairs Identified (>0.80): [('bathrooms', 'bedrooms'), ('price', 'area_sqft'), ('total_rooms', 'bedrooms'), ('yard_size_sqft', 'lot_size_sqft')]


In [163]:
# Step 4: Target Comparison against 'price'
corr_with_target = numeric_features.corr()["price"].abs()

# Vectorized Comparison
col1, col2 = high_corr_pairs[0]

# which has least correlation with target 'price' will be droped
feature_to_drop = col1 if corr_with_target[col1] < corr_with_target[col2] else col2

# Direct vectorized column deletion
df = df.drop(columns=[feature_to_drop], errors="ignore")

print(f"   [SEVERED] Systematically dropped '{feature_to_drop}' via vectorized matrix selection.")
print("\n=== PHASE 2 COMPLETED: ENGINE EXECUTED ===\n")
print("Remaining features in dataset:", df.columns.tolist())

   [SEVERED] Systematically dropped 'bathrooms' via vectorized matrix selection.

=== PHASE 2 COMPLETED: ENGINE EXECUTED ===

Remaining features in dataset: ['id', 'area_sqft', 'lot_size_sqft', 'bedrooms', 'stories', 'parking_spaces', 'house_age_years', 'distance_to_city_km', 'school_quality_index', 'crime_index', 'price', 'total_rooms', 'yard_size_sqft', 'space_per_room', 'area_FB Area', 'area_NayaNazimabad', 'area_NorthNazimabad', 'area_PECHS']


In [164]:
!pip install pandera

In [165]:
# PHASE 3 - TASK 1: RUNTIME SCHEMA CONTRACT (PANDERA)

import pandera.pandas as pa
from pandera.pandas import Check, Column, DataFrameSchema

In [166]:
# 1. Strict Enterprise Schema Design (Contracts)
# finalizing the rules for the remaining columns. These are the structural guidelines to ensure safety and stability after Phase 2

house_price_schema = DataFrameSchema(
    columns={
        "area_sqft": Column(float, Check.greater_than_or_equal_to(400.0)),
        "lot_size_sqft": Column(float, Check.greater_than(0.0)),
        "bathrooms": Column(
            int, Check.in_range(1, 10), required=False
        ),  # bedrooms drop hua tha to bathrooms safe hai
        "stories": Column(int, Check.in_range(1, 4)),
        "parking_spaces": Column(int, Check.greater_than_or_equal_to(0)),
        "house_age_years": Column(int, Check.greater_than_or_equal_to(0)),
        "distance_to_city_km": Column(float, Check.greater_than_or_equal_to(0.0)),
        "school_quality_index": Column(float, Check.in_range(1.0, 10.0)),
        "crime_index": Column(float, Check.in_range(1.0, 10.0)),
        "price": Column(float, Check.greater_than(0.0)),
        "total_rooms": Column(int, Check.greater_than(0)),
        "yard_size_sqft": Column(float),
        "space_per_room": Column(float, Check.greater_than(0.0)),
    },
    strict=False,  # dynamically added area OHE columns ko allow karne ke liye
)


In [167]:
# 2. Executing Data Interface Validation with Lazy Reporting
def validate_production_data(cleaned_df):
    try:
        # Enforcing contracts at runtime with lazy reporting enabled
        validated_df = house_price_schema.validate(cleaned_df, lazy=True)
        print(
            "PRODUCTION CONTRACT VALIDATED: Data architecture matches strict fidelity parameters!"
        )
        return validated_df
    except pa.errors.SchemaErrors as err:
        print("CRITICAL: Structural Contracts Violated!")
        print("\n--- Failure Cases Log ---")
        print(err.failure_cases[["column", "check", "failure_case"]].head())
        return None


# Validate our Phase 2 output DataFrame
df_validated = validate_production_data(df)

CRITICAL: Structural Contracts Violated!

--- Failure Cases Log ---
           column             check failure_case
0       area_sqft  dtype('float64')        int64
1   lot_size_sqft  dtype('float64')        int64
2  yard_size_sqft  dtype('float64')        int64
